# Module 17: spaCy Transformers (Detailed)


## 🤖 Using HuggingFace Transformers in spaCy

In 2017, the "Attention Is All You Need" paper introduced the Transformer architecture, rendering older CNN and RNN architectures largely obsolete for high-accuracy NLP tasks.

HuggingFace is the open-source hub for thousands of pre-trained Transformer models (like BERT, RoBERTa, DistilBERT, etc.).
The `spacy-transformers` library acts as a bridge, allowing you to drop ANY HuggingFace model directly into the base of a spaCy pipeline!


<br><br>

---

<br><br>


### 1. Installation & Environment Warnings

Transformers require massive tensor computations, which means they rely heavily on **PyTorch**. 

*⚠️ Warning: If you are running an extremely new version of Python (e.g., 3.13 or 3.14), PyTorch may not have pre-compiled binaries available yet. If you get installation errors, downgrade to Python 3.11 or 3.12.*


In [ ]:
"""
# To use transformers, you must install the specific package
!pip install spacy-transformers

# Download spaCy's pre-packaged Transformer model (uses RoBERTa-base)
!python -m spacy download en_core_web_trf
"""


<br><br>

---

<br><br>


### 2. Loading and Processing

Once installed, the Transformer model acts exactly like the CNN models (`en_core_web_sm`). 
The difference? It is significantly slower on a CPU, requires much more RAM, but is vastly more accurate at understanding complex context.


In [ ]:
import spacy
import warnings
warnings.filterwarnings("ignore")

print("Loading Transformer Model... (This takes longer than CNN models)")
try:
    nlp_trf = spacy.load("en_core_web_trf")
    
    # Notice this complex sentence with ambiguous phrasing
    text = "The Washington bank decided to bank with Washington on the new financial bill."
    doc = nlp_trf(text)
    
    print("\nEntities extracted by RoBERTa:")
    for ent in doc.ents:
        print(f"{ent.text:<15} | {ent.label_}")
        
except OSError:
    print("[ERROR] en_core_web_trf is not installed. Skipping execution.")
except Exception as e:
    print(f"[ERROR] Failed to load transformer: {e}")


<br><br>

---

<br><br>


### 3. Extracting Contextualized Embeddings (`trf_data`)

In Module 7, we learned about Static Word Vectors (`token.vector`). The major flaw is that the word "bank" (river) has the exact same mathematical vector as "bank" (finance).

Transformers generate **Contextualized Embeddings**. The mathematical representation of the word is calculated based on the entire sentence! You can access the raw PyTorch tensors produced by the HuggingFace model using the `doc._.trf_data` extension.

*Note: This is advanced deep-learning extraction, usually only necessary if you are passing the embeddings into a custom PyTorch model downstream.*


In [ ]:
try:
    # Only run if we successfully loaded the model above
    if 'nlp_trf' in locals():
        doc2 = nlp_trf("This is a test of contextual embeddings.")
        
        # Access the TransformerData object
        trf_data = doc2._.trf_data
        
        # trf_data.tensors contains lists of numpy arrays/PyTorch tensors
        # Index 0 is usually the hidden states of the tokens
        hidden_states = trf_data.tensors[0]
        
        print(f"Raw Hidden States Tensor Shape: {hidden_states.shape}")
        print("(Batch Size, Number of WordPieces, Hidden Dimension (usually 768))")
except Exception:
    pass


<br><br>

---

<br><br>


### 4. WordPieces vs. spaCy Tokens

HuggingFace Transformers don't tokenize text the same way spaCy does. They use Subword Tokenization (like WordPiece or Byte-Pair Encoding). 
For example, the word "unbelievable" might be split into `["un", "##believ", "##able"]`.

The `spacy-transformers` library handles the complex alignment between HuggingFace Subwords and spaCy Tokens automatically, mapping the subword tensors back to the correct linguistic tokens so `doc.ents` still works seamlessly!
